
# African High School Journal Platform MVP

This notebook recreates a complete Streamlit MVP repository for an AI-supported high-school research journal platform founded by Kavya. It includes:

- student manuscript upload and online writing;
- local AI-style plagiarism/similarity screening;
- AI reviewer matching by field and expertise;
- reviewer comments and publication recommendations;
- editor/admin publication workflow;
- searchable electronic publication library;
- a smoke test that validates the core workflow.

Run the cells from top to bottom. The notebook is designed for Google Colab, local Jupyter, or GitHub Codespaces.


In [ ]:

from pathlib import Path
import json
import shutil
import zipfile

PROJECT_NAME = "african_high_school_journal_platform"
PROJECT_FILES = {
  ".gitignore": "__pycache__/\n*.pyc\n.venv/\n.env\n*.db\n*.sqlite\n.DS_Store\n.streamlit/secrets.toml\ndata/journal_platform.db\n.pytest_cache/\n",
  ".streamlit/config.toml": "[theme]\nbase = \"light\"\nprimaryColor = \"#2E7D32\"\nbackgroundColor = \"#FFFFFF\"\nsecondaryBackgroundColor = \"#F4F8F4\"\ntextColor = \"#1E1E1E\"\n\n[server]\nmaxUploadSize = 20\n",
  "ARCHITECTURE.md": "# Architecture and Data Schema\n\n## Goal\n\nBuild a practical MVP for an electronic high-school journal platform focused on African students. The MVP supports submission, online drafting, AI-assisted plagiarism screening, reviewer matching, peer-review recommendations, and electronic publication.\n\n## High-level architecture\n\n```text\nBrowser\n  |\n  v\nStreamlit UI (app.py)\n  |\n  v\nBusiness services (journal_platform/services.py)\n  |\n  +--> AI similarity and matching (journal_platform/ai.py)\n  +--> File text extraction (journal_platform/document_io.py)\n  +--> SQLite persistence (journal_platform/db.py)\n  |\n  v\nSQLite database + optional CSV seed data\n```\n\n## Main entities\n\n### users\nStores students, reviewers, and admins. In this MVP, authentication is demo-only: a user enters name, email, role, country, and optional expertise.\n\n### manuscripts\nStores uploaded or online-written manuscripts, metadata, status, plagiarism summary, matched reviewer summary, and publication slug.\n\nStatus values:\n\n- draft\n- submitted\n- under_review\n- minor_revision\n- major_revision\n- accepted\n- rejected\n- published\n\n### manuscript_versions\nStores historical text snapshots. Each saved draft or submitted manuscript can be versioned.\n\n### review_assignments\nStores AI-matched reviewers and assignment state.\n\n### reviews\nStores reviewer recommendation and comments.\n\n### plagiarism_sources\nStores external or local corpus entries used for similarity checks. The demo seeds sample source texts from `data/sample_corpus.csv`.\n\n## AI logic\n\n### Plagiarism similarity\nThe MVP uses TF-IDF vectorization and cosine similarity against a local corpus. It reports the top matching sources and an overall similarity score. This is not a legal or academic misconduct determination; it is a revision aid.\n\n### Reviewer matching\nThe matcher builds a manuscript profile from title, field, abstract, keywords, and text. It compares that profile against reviewer expertise profiles using TF-IDF cosine similarity and adds a field match bonus.\n\n## Core workflow\n\n1. Student signs in with demo profile.\n2. Student uploads a file or writes in the online editor.\n3. AI plagiarism scan compares the submission against the local corpus.\n4. Student revises or submits.\n5. AI reviewer matcher assigns top reviewers.\n6. Reviewer enters portal, reads assignment, and leaves recommendation/comments.\n7. Editor/admin can update status and publish accepted papers.\n8. Published papers appear in the public library.\n\n## Security and governance limitations\n\nThis MVP is not production-secure. Before real student use, add:\n\n- real authentication and authorization;\n- parental/guardian consent process if required;\n- moderation and safeguarding policy;\n- encrypted storage and backups;\n- data retention and removal process;\n- reviewer conflict-of-interest disclosures;\n- audit logs;\n- terms of use and privacy policy;\n- accessibility review;\n- independent editorial board governance.\n",
  "LICENSE": "MIT License\n\nCopyright (c) 2026 Kavya and contributors\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the \"Software\"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED \"AS IS\", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n",
  "README.md": "# African High School Journal Platform\n\nA Streamlit MVP for an AI-supported high-school research journal platform founded by Kavya. The platform is designed to help African high school students upload manuscripts, write drafts online, receive AI-assisted plagiarism checks, match submissions to teachers/professors with relevant expertise, collect peer-review recommendations, and publish accepted electronic papers in an online library.\n\nThis repository is intentionally built with a free/low-cost stack:\n\n- Python 3.12+ recommended for Streamlit Community Cloud; Python 3.14 works locally if your dependencies support it.\n- Streamlit for the web interface.\n- SQLite for an MVP database.\n- scikit-learn TF-IDF similarity for local AI-style matching and plagiarism similarity.\n- pypdf and python-docx for PDF/DOCX text extraction.\n\n> Important: This MVP does not search the whole internet for plagiarism. It compares a manuscript to the platform's local corpus: published manuscripts, submitted manuscripts, and the optional sample/source corpus. For production-level internet plagiarism checks, integrate a licensed plagiarism/search API and add an appeals process.\n\n## Main functions\n\n1. **Student upload and online writing**\n   - Students can upload TXT, Markdown, PDF, or DOCX files.\n   - Students can also draft directly in the browser and save the manuscript online.\n\n2. **AI-based plagiarism detection**\n   - Uses local TF-IDF cosine similarity to compare the submission with the platform corpus.\n   - Shows matched source titles and similarity scores.\n   - Flags high similarity so students can revise before final submission.\n\n3. **AI-based reviewer matching**\n   - Matches manuscripts to reviewers based on field, keywords, abstract, manuscript text, and reviewer expertise.\n   - Creates reviewer assignments with shareable invite links.\n\n4. **Reviewer portal**\n   - Reviewers can view assigned manuscripts and submit recommendations: accept, minor revision, major revision, or reject.\n   - Review comments are stored with the submission.\n\n5. **Publication library**\n   - Accepted papers can be published electronically and shown in the searchable public library.\n\n## Repository structure\n\n```text\nafrican_high_school_journal_platform/\n  app.py                                # Streamlit web app\n  requirements.txt                      # Python dependencies\n  ARCHITECTURE.md                       # Architecture and schema explanation\n  schema.sql                            # SQLite schema\n  README.md                             # Setup and deployment guide\n  LICENSE                               # MIT license placeholder\n  .streamlit/config.toml                # Streamlit UI settings\n  data/\n    seed_reviewers.csv                  # Demo reviewer database\n    sample_corpus.csv                   # Demo comparison corpus\n  journal_platform/\n    __init__.py\n    ai.py                               # Similarity, plagiarism, reviewer matching\n    config.py                           # Configuration constants\n    db.py                               # SQLite helpers\n    document_io.py                      # PDF/DOCX/TXT extraction\n    services.py                         # Platform business logic\n  tests/\n    smoke_test.py                       # End-to-end smoke test\n  docs/\n    AUTHOR_GUIDELINES.md                # Student submission guidance\n    REVIEWER_GUIDELINES.md              # Reviewer criteria and tone\n    EDITORIAL_POLICY.md                 # Editorial workflow and safeguards\n    COPYRIGHT_AND_LICENSE_GUIDE.md      # Copyright/open-source notes\n    ROADMAP.md                          # MVP and production roadmap\n  notebooks/\n    high_school_journal_platform_colab.ipynb\n```\n\n## Run locally\n\n```bash\npython -m venv .venv\nsource .venv/bin/activate        # macOS/Linux\n# .venv\\Scripts\\activate.bat    # Windows Command Prompt\npython -m pip install -r requirements.txt\npython tests/smoke_test.py\nstreamlit run app.py\n```\n\n## Run in Google Colab\n\n1. Open `notebooks/high_school_journal_platform_colab.ipynb` in Google Colab.\n2. Run each cell from top to bottom.\n3. The notebook writes the project files, installs dependencies, runs the smoke test, and shows how to launch Streamlit.\n4. For a public demo link from Colab, use the included optional tunnel cell. Colab sessions are temporary, so this is for demos, not stable hosting.\n\n## Deploy on Streamlit Community Cloud\n\n1. Create a GitHub repository.\n2. Upload this project folder to the repository root.\n3. Go to Streamlit Community Cloud and create an app from your GitHub repository.\n4. Set the entrypoint file to `app.py`.\n5. Let Streamlit install dependencies from `requirements.txt`.\n\n## Suggested production upgrades\n\n- Replace demo login with a proper authentication provider.\n- Use PostgreSQL/Supabase/Firebase instead of SQLite for concurrent users.\n- Store manuscripts in cloud object storage.\n- Add consent, privacy, youth-safety, and data-retention policies for minors.\n- Add licensed plagiarism/search integration.\n- Add reviewer conflict-of-interest declarations.\n- Add editor workflow with multiple reviewers before final decision.\n- Add DOI/ISSN workflows only after formal journal governance is ready.\n\n## Copyright and open source note\n\nKavya and contributors can publish the source code under an open-source license such as MIT, Apache-2.0, or GPL. The platform name, logo, website content, and source code copyright notices should be kept clear. Individual student papers should keep author copyright unless the journal adopts a stated publication license.\n",
  "SMOKE_TEST_RESULTS.txt": "SMOKE TEST PASSED\nseeded: {'reviewers': 8, 'sources': 4}\nfirst_manuscript_id: 1\ncopy_similarity_score: 1.0\nreview_id: 1\npublished_slug: ahsj-2026-1-solar-panel-voltage-in-school-laboratories\nassignment_count: 6\n",
  "app.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\nimport streamlit as st\n\nfrom journal_platform.ai import plagiarism_scan, word_count\nfrom journal_platform.config import APP_NAME, APP_TAGLINE, DB_PATH, PLAGIARISM_HIGH_THRESHOLD, PLAGIARISM_LOW_THRESHOLD\nfrom journal_platform.db import execute, fetch_all, fetch_one, initialize_with_seed_data, upsert_user\nfrom journal_platform.document_io import read_uploaded_file\nfrom journal_platform.services import (\n    add_review,\n    get_assignments_for_reviewer,\n    get_local_similarity_corpus,\n    list_manuscripts,\n    list_published,\n    list_reviews,\n    manuscript_quality_hints,\n    manuscript_stats,\n    publish_manuscript,\n    save_draft,\n    setup_platform,\n    submit_manuscript,\n    update_status,\n)\n\nst.set_page_config(page_title=APP_NAME, page_icon=\"\ud83d\udcda\", layout=\"wide\")\n\n\n@st.cache_resource\ndef bootstrap_database() -> str:\n    initialize_with_seed_data(DB_PATH, reset=False)\n    return str(DB_PATH)\n\n\nbootstrap_database()\n\n\ndef show_status_badge(status: str) -> None:\n    status = status or \"unknown\"\n    icon = {\n        \"draft\": \"\ud83d\udcdd\",\n        \"submitted\": \"\ud83d\udce8\",\n        \"under_review\": \"\ud83d\udd0d\",\n        \"minor_revision\": \"\ud83d\udfe1\",\n        \"major_revision\": \"\ud83d\udfe0\",\n        \"accepted\": \"\u2705\",\n        \"rejected\": \"\u274c\",\n        \"published\": \"\ud83c\udf0d\",\n    }.get(status, \"\u2022\")\n    st.caption(f\"{icon} Status: `{status}`\")\n\n\ndef risk_box(score: float, report: str) -> None:\n    if score >= PLAGIARISM_HIGH_THRESHOLD:\n        st.error(report)\n    elif score >= PLAGIARISM_LOW_THRESHOLD:\n        st.warning(report)\n    else:\n        st.success(report)\n\n\nwith st.sidebar:\n    st.title(\"Journal login\")\n    role = st.selectbox(\"Role\", [\"student\", \"reviewer\", \"admin\"])\n    name = st.text_input(\"Name\", value=\"Kavya\" if role == \"admin\" else \"\")\n    email = st.text_input(\"Email\", value=\"kavya@example.org\" if role == \"admin\" else \"\")\n    country = st.text_input(\"Country\", value=\"\")\n    expertise = \"\"\n    if role == \"reviewer\":\n        expertise = st.text_area(\"Reviewer expertise\", placeholder=\"Example: solar energy, circuits, engineering education\")\n    if st.button(\"Save demo profile\"):\n        if not name or not email:\n            st.error(\"Please enter name and email.\")\n        else:\n            user_id = upsert_user(DB_PATH, name, email, role, country=country, expertise=expertise or None)\n            st.session_state[\"user\"] = {\"user_id\": user_id, \"name\": name, \"email\": email, \"role\": role, \"country\": country}\n            st.success(f\"Profile saved for {name}.\")\n\n    st.divider()\n    stats = manuscript_stats(DB_PATH)\n    st.metric(\"Published\", stats.get(\"published\", 0))\n    st.metric(\"Under review\", stats.get(\"under_review\", 0))\n    st.metric(\"Reviewers\", stats.get(\"reviewers\", 0))\n\nst.title(APP_NAME)\nst.caption(APP_TAGLINE)\nst.info(\n    \"MVP note: this demo uses local similarity search and demo login. For real student use, add production authentication, privacy/consent policies, and a licensed plagiarism source.\"\n)\n\nhome_tab, submit_tab, review_tab, admin_tab, about_tab = st.tabs(\n    [\"Published Library\", \"Student Submit / Write\", \"Reviewer Portal\", \"Editor Admin\", \"About\"]\n)\n\nwith home_tab:\n    st.header(\"Published electronic papers\")\n    query = st.text_input(\"Search title, abstract, author, or keywords\", key=\"library_search\")\n    papers = list_published(DB_PATH, query=query)\n    if not papers:\n        st.write(\"No published papers yet. Use the admin tab to publish an accepted manuscript.\")\n    for paper in papers:\n        with st.expander(f\"{paper['title']} \u2014 {paper.get('author_name', 'Unknown author')}\", expanded=False):\n            show_status_badge(paper[\"status\"])\n            st.write(f\"**Author:** {paper.get('author_name', '')} ({paper.get('country', '')})\")\n            st.write(f\"**Field:** {paper.get('field', '')}\")\n            st.write(f\"**Keywords:** {paper.get('keywords', '')}\")\n            st.write(f\"**Publication slug:** `{paper.get('doi_slug', '')}`\")\n            st.write(\"**Abstract**\")\n            st.write(paper.get(\"abstract\", \"\"))\n            st.write(\"**Manuscript text**\")\n            st.write(paper.get(\"manuscript_text\", \"\")[:6000])\n\nwith submit_tab:\n    st.header(\"Student manuscript submission\")\n    st.write(\"Upload a manuscript or write directly in the online editor. Run a local AI similarity screen before submission.\")\n    col_a, col_b = st.columns(2)\n    with col_a:\n        student_name = st.text_input(\"Student author name\", value=st.session_state.get(\"user\", {}).get(\"name\", \"\"), key=\"student_name\")\n        student_email = st.text_input(\"Student email\", value=st.session_state.get(\"user\", {}).get(\"email\", \"\"), key=\"student_email\")\n        student_country = st.text_input(\"Country\", value=st.session_state.get(\"user\", {}).get(\"country\", \"\"), key=\"student_country\")\n        title = st.text_input(\"Manuscript title\")\n        field = st.selectbox(\n            \"Research field\",\n            [\n                \"Biology\",\n                \"Chemistry\",\n                \"Computer Science\",\n                \"Data Science\",\n                \"Engineering\",\n                \"Energy\",\n                \"Environment\",\n                \"Mathematics\",\n                \"Physics\",\n                \"Public Health\",\n                \"Robotics\",\n                \"Social Science\",\n                \"Education\",\n                \"Other\",\n            ],\n        )\n        keywords = st.text_input(\"Keywords, separated by commas\")\n    with col_b:\n        abstract = st.text_area(\"Abstract\", height=180)\n        uploaded_file = st.file_uploader(\"Upload manuscript file\", type=[\"txt\", \"md\", \"pdf\", \"docx\"])\n\n    uploaded_text = \"\"\n    file_name = \"online-editor\"\n    if uploaded_file is not None:\n        try:\n            uploaded_text = read_uploaded_file(uploaded_file)\n            file_name = uploaded_file.name\n            st.success(f\"Extracted {word_count(uploaded_text)} words from {uploaded_file.name}.\")\n        except Exception as exc:\n            st.error(f\"Could not read uploaded file: {exc}\")\n\n    manuscript_text = st.text_area(\n        \"Online manuscript editor\",\n        value=uploaded_text,\n        height=420,\n        placeholder=\"Paste or write the full manuscript here. Suggested sections: Abstract, Introduction, Methods, Results, Discussion, References.\",\n    )\n    st.caption(f\"Current manuscript length: {word_count(manuscript_text)} words\")\n\n    hints = manuscript_quality_hints(manuscript_text)\n    if hints:\n        with st.expander(\"AI writing and structure hints\"):\n            for hint in hints:\n                st.write(f\"- {hint}\")\n\n    scan_result = None\n    if st.button(\"Run local AI plagiarism / similarity scan\"):\n        corpus = get_local_similarity_corpus(DB_PATH)\n        scan_result = plagiarism_scan(manuscript_text, corpus, top_n=5)\n        st.session_state[\"last_scan\"] = scan_result\n    if \"last_scan\" in st.session_state:\n        scan_result = st.session_state[\"last_scan\"]\n        risk_box(scan_result[\"score\"], scan_result[\"report\"])\n        st.json(scan_result[\"matches\"], expanded=False)\n\n    col1, col2 = st.columns(2)\n    with col1:\n        if st.button(\"Save as online draft\"):\n            required = [student_name, student_email, title, field, manuscript_text]\n            if not all(required):\n                st.error(\"Please complete name, email, title, field, and manuscript text.\")\n            else:\n                manuscript_id = save_draft(\n                    DB_PATH,\n                    student_name,\n                    student_email,\n                    student_country,\n                    title,\n                    abstract,\n                    field,\n                    keywords,\n                    manuscript_text,\n                    file_name=file_name,\n                )\n                st.success(f\"Draft saved. Manuscript ID: {manuscript_id}\")\n    with col2:\n        if st.button(\"Submit for review\"):\n            required = [student_name, student_email, title, field, manuscript_text]\n            if not all(required):\n                st.error(\"Please complete name, email, title, field, and manuscript text.\")\n            else:\n                result = submit_manuscript(\n                    DB_PATH,\n                    student_name,\n                    student_email,\n                    student_country,\n                    title,\n                    abstract,\n                    field,\n                    keywords,\n                    manuscript_text,\n                    file_name=file_name,\n                    plagiarism_override=st.session_state.get(\"last_scan\"),\n                )\n                st.success(f\"Submitted manuscript ID {result['manuscript_id']} and assigned reviewers.\")\n                risk_box(result[\"plagiarism\"][\"score\"], result[\"plagiarism\"][\"report\"])\n                if result[\"assignments\"]:\n                    st.write(\"**AI-matched reviewer assignments**\")\n                    st.dataframe(pd.DataFrame(result[\"assignments\"])[[\"name\", \"email\", \"country\", \"match_score\", \"assignment_link\"]])\n\nwith review_tab:\n    st.header(\"Reviewer portal\")\n    reviewer_email = st.text_input(\"Reviewer email\", value=st.session_state.get(\"user\", {}).get(\"email\", \"\"), key=\"reviewer_email\")\n    reviewer_name = st.text_input(\"Reviewer name\", value=st.session_state.get(\"user\", {}).get(\"name\", \"\"), key=\"reviewer_name\")\n    if reviewer_email:\n        assignments = get_assignments_for_reviewer(DB_PATH, reviewer_email)\n        if not assignments:\n            st.warning(\"No assignments found for this email. Admins can add reviewers or submit a manuscript to trigger matching.\")\n        for assignment in assignments:\n            with st.expander(f\"Review: {assignment['title']} ({assignment['manuscript_status']})\", expanded=False):\n                st.write(f\"**Author:** {assignment['author_name']} ({assignment['country']})\")\n                st.write(f\"**Field:** {assignment['field']}\")\n                st.write(f\"**Keywords:** {assignment['keywords']}\")\n                st.write(f\"**Match score:** {assignment['match_score']:.1%}\")\n                st.write(\"**Abstract**\")\n                st.write(assignment[\"abstract\"])\n                st.write(\"**Manuscript text**\")\n                st.text_area(\"Read manuscript\", assignment[\"manuscript_text\"], height=260, key=f\"read_{assignment['assignment_id']}\")\n                recommendation = st.selectbox(\n                    \"Recommendation\",\n                    [\"accept\", \"minor_revision\", \"major_revision\", \"reject\"],\n                    key=f\"rec_{assignment['assignment_id']}\",\n                )\n                comments_author = st.text_area(\"Comments to author\", key=f\"author_comments_{assignment['assignment_id']}\")\n                comments_editor = st.text_area(\"Confidential comments to editor\", key=f\"editor_comments_{assignment['assignment_id']}\")\n                confidence = st.slider(\"Reviewer confidence\", 1, 5, 3, key=f\"confidence_{assignment['assignment_id']}\")\n                if st.button(\"Submit review\", key=f\"submit_review_{assignment['assignment_id']}\"):\n                    if not reviewer_name or not reviewer_email or not comments_author:\n                        st.error(\"Reviewer name, email, and author comments are required.\")\n                    else:\n                        review_id = add_review(\n                            DB_PATH,\n                            int(assignment[\"manuscript_id\"]),\n                            reviewer_name,\n                            reviewer_email,\n                            recommendation,\n                            comments_author,\n                            comments_editor,\n                            confidence,\n                        )\n                        st.success(f\"Review submitted. Review ID: {review_id}\")\n\nwith admin_tab:\n    st.header(\"Editor/admin dashboard\")\n    st.write(\"For the MVP, admin access is not secured. Add real authorization before public use.\")\n    if st.button(\"Initialize/seed demo reviewers and sample corpus\"):\n        seeded = setup_platform(DB_PATH, reset=False)\n        st.success(f\"Seed complete: {seeded}\")\n\n    with st.expander(\"Add a reviewer\"):\n        r_name = st.text_input(\"Reviewer full name\", key=\"admin_reviewer_name\")\n        r_email = st.text_input(\"Reviewer email\", key=\"admin_reviewer_email\")\n        r_country = st.text_input(\"Reviewer country\", key=\"admin_reviewer_country\")\n        r_expertise = st.text_area(\"Expertise profile\", key=\"admin_reviewer_expertise\")\n        if st.button(\"Add/update reviewer\"):\n            if r_name and r_email and r_expertise:\n                upsert_user(DB_PATH, r_name, r_email, \"reviewer\", country=r_country, expertise=r_expertise)\n                st.success(\"Reviewer saved.\")\n            else:\n                st.error(\"Name, email, and expertise are required.\")\n\n    manuscripts = list_manuscripts(DB_PATH)\n    if manuscripts:\n        df = pd.DataFrame(manuscripts)\n        st.dataframe(df[[\"manuscript_id\", \"title\", \"author_name\", \"country\", \"field\", \"status\", \"plagiarism_score\", \"matched_reviewers\"]])\n        selected_id = st.number_input(\"Manuscript ID to manage\", min_value=1, step=1)\n        manuscript = fetch_one(DB_PATH, \"SELECT * FROM manuscripts WHERE manuscript_id = ?\", (int(selected_id),))\n        if manuscript:\n            st.subheader(manuscript[\"title\"])\n            show_status_badge(manuscript[\"status\"])\n            st.write(\"**Plagiarism report**\")\n            st.text(manuscript[\"plagiarism_report\"])\n            st.write(\"**Matched reviewers**\")\n            st.write(manuscript[\"matched_reviewers\"] or \"No matches stored.\")\n            reviews = list_reviews(DB_PATH, int(selected_id))\n            if reviews:\n                st.write(\"**Reviews**\")\n                st.dataframe(pd.DataFrame(reviews)[[\"reviewer_name\", \"recommendation\", \"confidence\", \"comments_to_author\", \"created_at\"]])\n            new_status = st.selectbox(\n                \"Set status\",\n                [\"draft\", \"submitted\", \"under_review\", \"minor_revision\", \"major_revision\", \"accepted\", \"rejected\", \"published\"],\n                index=[\"draft\", \"submitted\", \"under_review\", \"minor_revision\", \"major_revision\", \"accepted\", \"rejected\", \"published\"].index(manuscript[\"status\"]),\n            )\n            col_s, col_p = st.columns(2)\n            with col_s:\n                if st.button(\"Update status\"):\n                    update_status(DB_PATH, int(selected_id), new_status)\n                    st.success(\"Status updated.\")\n            with col_p:\n                if st.button(\"Publish manuscript electronically\"):\n                    slug = publish_manuscript(DB_PATH, int(selected_id))\n                    st.success(f\"Published with slug: {slug}\")\n    else:\n        st.write(\"No manuscripts yet.\")\n\n    with st.expander(\"Raw data export\"):\n        if manuscripts:\n            st.download_button(\n                \"Download manuscript CSV\",\n                pd.DataFrame(manuscripts).to_csv(index=False).encode(\"utf-8\"),\n                file_name=\"manuscripts_export.csv\",\n                mime=\"text/csv\",\n            )\n\nwith about_tab:\n    st.header(\"About this MVP\")\n    st.markdown(\n        \"\"\"\n        **Mission.** Encourage African high school students to practice responsible research, peer review, revision, and electronic publication.\n\n        **Founder/build lead.** Kavya can use this as a GitHub-open-source MVP and build toward a formal journal platform.\n\n        **Electronic first.** This app publishes papers electronically. Printed hard copies can be considered later with a publishing/printing partner.\n\n        **Ethical safeguards.** Because students may be minors, production use should include guardian consent, data privacy, content moderation, safe communication rules, and reviewer conflict-of-interest policies.\n\n        **AI transparency.** The similarity scanner and reviewer matcher are assistive tools. Editors and teachers should make final decisions.\n        \"\"\"\n    )\n",
  "data/sample_corpus.csv": "title,source_type,url,source_text\nDemo Source - Solar Cooking,seed,,\"Solar cooking research often studies reflector geometry, insulation, thermal storage, and cooking efficiency under different sunlight conditions. Student projects may compare temperatures reached by box cookers and parabolic cookers.\"\nDemo Source - Water Quality,seed,,\"Water quality investigations commonly measure pH, turbidity, nitrate concentration, and microbial contamination. Research designs often compare water samples across neighborhoods and discuss public health implications.\"\nDemo Source - Machine Learning Education,seed,,\"Machine learning education projects introduce supervised learning, data preprocessing, train test splitting, model evaluation, and ethical use of data. Simple classifiers can help students explore real world prediction problems.\"\nDemo Source - Robotics Sensors,seed,,\"Robotics projects use sensors, microcontrollers, motors, feedback loops, and control algorithms. A typical high school robotics manuscript describes the design process, testing results, limitations, and future improvements.\"\n",
  "data/seed_reviewers.csv": "name,email,country,expertise,fields\nDr. Amina Diallo,amina.diallo@example.org,Senegal,\"renewable energy, solar cells, physics education, electrical circuits\",Physics;Engineering;Energy\nProf. Kwame Mensah,kwame.mensah@example.org,Ghana,\"machine learning, data science, statistics, Python, youth research mentoring\",Computer Science;Data Science;Mathematics\nDr. Njeri Okafor,njeri.okafor@example.org,Kenya,\"public health, biology, epidemiology, scientific writing for students\",Biology;Public Health\nMs. Thandiwe Moyo,thandiwe.moyo@example.org,South Africa,\"literature review methods, social science research, education policy\",Social Science;Education\nDr. Youssef Benali,youssef.benali@example.org,Morocco,\"robotics, embedded systems, control systems, engineering design\",Engineering;Robotics\nDr. Fatima Hassan,fatima.hassan@example.org,Egypt,\"environmental chemistry, water quality, climate adaptation, laboratory methods\",Chemistry;Environment\nMr. Samuel Adeyemi,samuel.adeyemi@example.org,Nigeria,\"mathematics modeling, optimization, statistics, student research projects\",Mathematics;Data Science\nDr. Mariam Kamara,mariam.kamara@example.org,Sierra Leone,\"agriculture technology, food security, remote sensing, sustainable development\",Agriculture;Environment\n",
  "docs/AUTHOR_GUIDELINES.md": "# Author Guidelines\n\n## Scope\n\nThe African High School Research Journal welcomes original student research manuscripts from high school students, with a special mission to encourage African students to participate in responsible research and publication.\n\n## Submission types\n\n- Original research article\n- Short research communication\n- Review article\n- Engineering or prototype report\n- Data science project report\n- Community research report\n\n## Suggested manuscript structure\n\n1. Title\n2. Author name, school, country, and contact email\n3. Abstract, 150 to 250 words\n4. Keywords, 3 to 8 terms\n5. Introduction and research question\n6. Methods or design process\n7. Results\n8. Discussion\n9. Limitations\n10. Conclusion\n11. References\n12. Acknowledgments, if any\n\n## Academic integrity\n\nStudents must write manuscripts in their own words. Any copied ideas, figures, data, or quotations must be cited. The platform's local AI similarity screen is a revision aid, not a final misconduct judgment.\n\n## Ethics and safety\n\nProjects involving human participants, medical information, interviews, surveys, hazardous materials, or sensitive data should include teacher supervision and follow applicable school and local rules.\n\n## Publication model\n\nThis MVP supports electronic publication first. Printed issues may be considered later if the journal has sufficient demand and funding.\n",
  "docs/COPYRIGHT_AND_LICENSE_GUIDE.md": "# Copyright and License Guide\n\nThis guide is educational and is not legal advice.\n\n## Source code\n\nKavya and contributors can publish the platform code on GitHub under an open-source license. The included `LICENSE` file uses the MIT License as a simple default. Other choices include Apache-2.0 or GPL, depending on the project's goals.\n\n## Website name and branding\n\nThe platform name, logo, and branding should be documented. If a formal trademark strategy is needed, consult a qualified professional.\n\n## Student papers\n\nStudent authors should normally retain copyright in their own manuscripts unless they sign a clear publication agreement. The journal can request permission to publish electronically.\n\n## Recommended publication license options\n\n- All rights reserved by the student author, with permission for the journal to publish.\n- Creative Commons Attribution, if authors want reuse with credit.\n- Creative Commons Attribution-NonCommercial, if authors want noncommercial reuse with credit.\n\n## GitHub release checklist\n\n1. Add copyright notice in README and source files if desired.\n2. Keep the LICENSE file in the repository root.\n3. Add contributor guidelines before accepting outside code.\n4. Keep student manuscript data out of the public GitHub repository unless authors consent.\n5. Use GitHub releases for versioned source-code packages.\n",
  "docs/EDITORIAL_POLICY.md": "# Editorial Policy\n\n## Mission\n\nThe journal promotes high school research literacy, peer review, and publication opportunities, especially for African students and schools.\n\n## Editorial workflow\n\n1. Submission is received.\n2. Local AI similarity screening is shown to the student/editor.\n3. The platform suggests reviewers based on expertise similarity.\n4. Reviewers submit recommendations and comments.\n5. Editors make a final decision.\n6. Accepted manuscripts are published electronically.\n\n## Editorial independence\n\nAI tools may recommend reviewers and flag similarity, but editors and teachers make final decisions.\n\n## Corrections and withdrawals\n\nThe journal should maintain a process for correcting published manuscripts and withdrawing papers when serious issues are discovered.\n\n## Data protection\n\nBefore real student use, the journal should publish a privacy policy, define retention periods, limit collection of minor data, and obtain consent where needed.\n\n## Accessibility\n\nThe platform should aim to support low-bandwidth access, readable formatting, mobile devices, and accessibility best practices.\n",
  "docs/REVIEWER_GUIDELINES.md": "# Reviewer Guidelines\n\n## Reviewer role\n\nReviewers help students improve research quality, clarity, ethics, and presentation. The goal is educational peer review, not gatekeeping.\n\n## Recommendations\n\n- Accept: suitable for publication with only editorial polish.\n- Minor revision: publishable after small corrections.\n- Major revision: promising but needs substantial revision.\n- Reject: not suitable in current form, or outside scope, or contains major integrity/safety issues.\n\n## Review criteria\n\n1. Originality and student authorship\n2. Clarity of research question\n3. Appropriateness of methods\n4. Quality of data or evidence\n5. Interpretation of results\n6. Ethical and safety considerations\n7. Writing quality and structure\n8. Correct use of citations\n9. Constructive feedback for improvement\n\n## Conflict of interest\n\nReviewers should decline assignments when they have a personal, financial, supervisory, or institutional conflict that could affect fairness.\n\n## Student-centered tone\n\nComments should be kind, specific, educational, and actionable. Avoid harsh language.\n",
  "docs/ROADMAP.md": "# Roadmap\n\n## MVP included here\n\n- Streamlit web app\n- SQLite database\n- Student upload and online editor\n- Local AI similarity screen\n- AI reviewer matching\n- Reviewer portal\n- Editor/admin publication tools\n- Public electronic library\n- Smoke test\n- Colab notebook\n\n## Next technical milestones\n\n1. Authentication with roles and secure sessions\n2. Cloud database, such as PostgreSQL or Supabase\n3. Cloud file storage\n4. Email invitations for reviewers\n5. Multiple-reviewer decision logic\n6. Full-text search index\n7. DOI/ISSN exploration after governance is established\n8. Accessibility audit\n9. Production logging and backups\n10. Licensed plagiarism/search API integration\n\n## Governance milestones\n\n1. Editorial board\n2. Reviewer recruitment and verification\n3. Privacy and child-safety policies\n4. Author publication agreement\n5. Conflict-of-interest process\n6. Corrections and withdrawals process\n7. Partnerships with schools and teachers\n",
  "journal_platform/__init__.py": "\"\"\"African High School Journal Platform MVP package.\"\"\"\n\n__all__ = [\"ai\", \"config\", \"db\", \"document_io\", \"services\"]\n__version__ = \"0.1.0\"\n",
  "journal_platform/ai.py": "from __future__ import annotations\n\nimport re\nfrom dataclasses import dataclass\nfrom typing import Iterable\n\nimport numpy as np\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\n\nSTOP_NOTICE = \"This is a local similarity screen, not a misconduct judgment.\"\n\n\ndef normalize_text(text: str) -> str:\n    text = text or \"\"\n    text = re.sub(r\"\\s+\", \" \", text)\n    return text.strip()\n\n\ndef word_count(text: str) -> int:\n    return len(re.findall(r\"\\b\\w+\\b\", text or \"\"))\n\n\ndef safe_vectorizer(max_features: int = 5000) -> TfidfVectorizer:\n    return TfidfVectorizer(\n        lowercase=True,\n        stop_words=\"english\",\n        ngram_range=(1, 2),\n        max_features=max_features,\n        min_df=1,\n    )\n\n\n@dataclass\nclass SimilarityMatch:\n    title: str\n    similarity: float\n    source_type: str = \"local\"\n    url: str = \"\"\n\n\ndef similarity_rankings(candidate_text: str, corpus: Iterable[dict], top_n: int = 5) -> list[SimilarityMatch]:\n    candidate_text = normalize_text(candidate_text)\n    records = [r for r in corpus if normalize_text(str(r.get(\"source_text\", \"\") or r.get(\"manuscript_text\", \"\")))]\n    if not candidate_text or not records:\n        return []\n\n    corpus_texts = [normalize_text(str(r.get(\"source_text\", \"\") or r.get(\"manuscript_text\", \"\"))) for r in records]\n    vectorizer = safe_vectorizer()\n    try:\n        matrix = vectorizer.fit_transform([candidate_text] + corpus_texts)\n    except ValueError:\n        return []\n    sims = cosine_similarity(matrix[0:1], matrix[1:]).ravel()\n    order = np.argsort(sims)[::-1][:top_n]\n    matches: list[SimilarityMatch] = []\n    for idx in order:\n        record = records[int(idx)]\n        matches.append(\n            SimilarityMatch(\n                title=str(record.get(\"title\") or f\"Source {idx + 1}\"),\n                similarity=float(sims[int(idx)]),\n                source_type=str(record.get(\"source_type\") or record.get(\"status\") or \"local\"),\n                url=str(record.get(\"url\") or \"\"),\n            )\n        )\n    return matches\n\n\ndef plagiarism_scan(candidate_text: str, corpus: Iterable[dict], top_n: int = 5) -> dict:\n    matches = similarity_rankings(candidate_text, corpus, top_n=top_n)\n    score = max([m.similarity for m in matches], default=0.0)\n    if score >= 0.45:\n        risk = \"high\"\n    elif score >= 0.20:\n        risk = \"medium\"\n    else:\n        risk = \"low\"\n    report_lines = [STOP_NOTICE, f\"Overall local similarity risk: {risk.upper()} ({score:.1%}).\"]\n    if matches:\n        report_lines.append(\"Top local matches:\")\n        for m in matches:\n            report_lines.append(f\"- {m.title}: {m.similarity:.1%} similarity ({m.source_type})\")\n    else:\n        report_lines.append(\"No local corpus matches were available.\")\n    return {\n        \"score\": float(score),\n        \"risk\": risk,\n        \"matches\": [m.__dict__ for m in matches],\n        \"report\": \"\\n\".join(report_lines),\n    }\n\n\ndef manuscript_profile(title: str, abstract: str, field: str, keywords: str, manuscript_text: str) -> str:\n    return normalize_text(\" \".join([title or \"\", abstract or \"\", field or \"\", keywords or \"\", manuscript_text or \"\"]))\n\n\ndef match_reviewers(\n    title: str,\n    abstract: str,\n    field: str,\n    keywords: str,\n    manuscript_text: str,\n    reviewers: Iterable[dict],\n    top_n: int = 3,\n) -> list[dict]:\n    reviewer_records = [r for r in reviewers if normalize_text(str(r.get(\"expertise\", \"\")))]\n    if not reviewer_records:\n        return []\n\n    profile = manuscript_profile(title, abstract, field, keywords, manuscript_text)\n    reviewer_texts = [normalize_text(str(r.get(\"expertise\", \"\"))) for r in reviewer_records]\n    vectorizer = safe_vectorizer(max_features=4000)\n    try:\n        matrix = vectorizer.fit_transform([profile] + reviewer_texts)\n    except ValueError:\n        return []\n    sims = cosine_similarity(matrix[0:1], matrix[1:]).ravel()\n    field_terms = {x.strip().lower() for x in re.split(r\"[,;/]\", field or \"\") if x.strip()}\n    scored = []\n    for idx, reviewer in enumerate(reviewer_records):\n        expertise = str(reviewer.get(\"expertise\", \"\")).lower()\n        field_bonus = 0.10 if any(term and term in expertise for term in field_terms) else 0.0\n        score = min(float(sims[idx]) + field_bonus, 1.0)\n        scored.append(\n            {\n                \"user_id\": reviewer.get(\"user_id\"),\n                \"name\": reviewer.get(\"name\"),\n                \"email\": reviewer.get(\"email\"),\n                \"country\": reviewer.get(\"country\"),\n                \"expertise\": reviewer.get(\"expertise\"),\n                \"match_score\": score,\n            }\n        )\n    scored.sort(key=lambda x: x[\"match_score\"], reverse=True)\n    return scored[:top_n]\n\n\ndef extract_top_terms(text: str, limit: int = 8) -> list[str]:\n    text = normalize_text(text)\n    if not text:\n        return []\n    vectorizer = safe_vectorizer(max_features=1000)\n    try:\n        matrix = vectorizer.fit_transform([text])\n    except ValueError:\n        return []\n    scores = matrix.toarray()[0]\n    terms = np.array(vectorizer.get_feature_names_out())\n    if scores.size == 0:\n        return []\n    order = np.argsort(scores)[::-1]\n    return [str(terms[i]) for i in order[:limit] if scores[i] > 0]\n",
  "journal_platform/config.py": "from pathlib import Path\nimport os\n\nAPP_NAME = \"African High School Research Journal\"\nAPP_TAGLINE = \"AI-supported student research submission, review, and publication\"\n\nBASE_DIR = Path(__file__).resolve().parents[1]\nDATA_DIR = BASE_DIR / \"data\"\nDATA_DIR.mkdir(exist_ok=True)\n\nDB_PATH = Path(os.getenv(\"JOURNAL_DB_PATH\", DATA_DIR / \"journal_platform.db\"))\nSCHEMA_PATH = BASE_DIR / \"schema.sql\"\nSEED_REVIEWERS_PATH = DATA_DIR / \"seed_reviewers.csv\"\nSAMPLE_CORPUS_PATH = DATA_DIR / \"sample_corpus.csv\"\n\nPLAGIARISM_LOW_THRESHOLD = 0.20\nPLAGIARISM_HIGH_THRESHOLD = 0.45\nTOP_REVIEWERS = 3\n",
  "journal_platform/db.py": "from __future__ import annotations\n\nimport csv\nimport sqlite3\nfrom pathlib import Path\nfrom typing import Any, Iterable\n\nfrom .config import DB_PATH, SCHEMA_PATH, SEED_REVIEWERS_PATH, SAMPLE_CORPUS_PATH\n\n\ndef get_connection(db_path: Path | str = DB_PATH) -> sqlite3.Connection:\n    conn = sqlite3.connect(str(db_path), check_same_thread=False)\n    conn.row_factory = sqlite3.Row\n    conn.execute(\"PRAGMA foreign_keys = ON\")\n    return conn\n\n\ndef initialize_database(db_path: Path | str = DB_PATH, reset: bool = False) -> None:\n    db_path = Path(db_path)\n    db_path.parent.mkdir(parents=True, exist_ok=True)\n    if reset and db_path.exists():\n        db_path.unlink()\n    with get_connection(db_path) as conn:\n        schema_sql = SCHEMA_PATH.read_text(encoding=\"utf-8\")\n        conn.executescript(schema_sql)\n        conn.commit()\n\n\ndef execute(db_path: Path | str, sql: str, params: Iterable[Any] = ()) -> int:\n    with get_connection(db_path) as conn:\n        cur = conn.execute(sql, tuple(params))\n        conn.commit()\n        return int(cur.lastrowid or 0)\n\n\ndef fetch_all(db_path: Path | str, sql: str, params: Iterable[Any] = ()) -> list[dict[str, Any]]:\n    with get_connection(db_path) as conn:\n        rows = conn.execute(sql, tuple(params)).fetchall()\n    return [dict(row) for row in rows]\n\n\ndef fetch_one(db_path: Path | str, sql: str, params: Iterable[Any] = ()) -> dict[str, Any] | None:\n    rows = fetch_all(db_path, sql, params)\n    return rows[0] if rows else None\n\n\ndef upsert_user(\n    db_path: Path | str,\n    name: str,\n    email: str,\n    role: str,\n    country: str | None = None,\n    expertise: str | None = None,\n) -> int:\n    email = email.strip().lower()\n    existing = fetch_one(db_path, \"SELECT user_id FROM users WHERE email = ?\", (email,))\n    if existing:\n        execute(\n            db_path,\n            \"\"\"\n            UPDATE users\n            SET name = ?, role = ?, country = COALESCE(?, country), expertise = COALESCE(?, expertise)\n            WHERE email = ?\n            \"\"\",\n            (name.strip(), role, country, expertise, email),\n        )\n        return int(existing[\"user_id\"])\n    return execute(\n        db_path,\n        \"\"\"\n        INSERT INTO users (name, email, role, country, expertise)\n        VALUES (?, ?, ?, ?, ?)\n        \"\"\",\n        (name.strip(), email, role, country, expertise),\n    )\n\n\ndef seed_reviewers(db_path: Path | str, csv_path: Path | str = SEED_REVIEWERS_PATH) -> int:\n    csv_path = Path(csv_path)\n    if not csv_path.exists():\n        return 0\n    count = 0\n    with csv_path.open(\"r\", encoding=\"utf-8\", newline=\"\") as f:\n        for row in csv.DictReader(f):\n            fields = row.get(\"fields\", \"\")\n            expertise = row.get(\"expertise\", \"\")\n            combined = f\"{expertise}; fields: {fields}\".strip()\n            upsert_user(\n                db_path,\n                name=row.get(\"name\", \"Reviewer\").strip(),\n                email=row.get(\"email\", \"\").strip(),\n                role=\"reviewer\",\n                country=row.get(\"country\", \"\").strip(),\n                expertise=combined,\n            )\n            count += 1\n    return count\n\n\ndef seed_sample_corpus(db_path: Path | str, csv_path: Path | str = SAMPLE_CORPUS_PATH) -> int:\n    csv_path = Path(csv_path)\n    if not csv_path.exists():\n        return 0\n    inserted = 0\n    with csv_path.open(\"r\", encoding=\"utf-8\", newline=\"\") as f:\n        for row in csv.DictReader(f):\n            title = row.get(\"title\", \"Untitled source\").strip()\n            text = row.get(\"source_text\", \"\").strip()\n            if not text:\n                continue\n            existing = fetch_one(db_path, \"SELECT source_id FROM plagiarism_sources WHERE title = ?\", (title,))\n            if existing:\n                continue\n            execute(\n                db_path,\n                \"\"\"\n                INSERT INTO plagiarism_sources (title, source_type, source_text, url)\n                VALUES (?, ?, ?, ?)\n                \"\"\",\n                (title, row.get(\"source_type\", \"seed\"), text, row.get(\"url\", \"\")),\n            )\n            inserted += 1\n    return inserted\n\n\ndef initialize_with_seed_data(db_path: Path | str = DB_PATH, reset: bool = False) -> dict[str, int]:\n    initialize_database(db_path, reset=reset)\n    reviewers = seed_reviewers(db_path)\n    sources = seed_sample_corpus(db_path)\n    return {\"reviewers\": reviewers, \"sources\": sources}\n",
  "journal_platform/document_io.py": "from __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nfrom typing import BinaryIO\n\n\ndef _read_pdf_bytes(data: bytes) -> str:\n    from pypdf import PdfReader\n\n    reader = PdfReader(BytesIO(data))\n    pages = []\n    for page in reader.pages:\n        pages.append(page.extract_text() or \"\")\n    return \"\\n\".join(pages).strip()\n\n\ndef _read_docx_bytes(data: bytes) -> str:\n    from docx import Document\n\n    doc = Document(BytesIO(data))\n    return \"\\n\".join(p.text for p in doc.paragraphs).strip()\n\n\ndef extract_text_from_bytes(data: bytes, filename: str) -> str:\n    suffix = Path(filename).suffix.lower()\n    if suffix == \".pdf\":\n        return _read_pdf_bytes(data)\n    if suffix == \".docx\":\n        return _read_docx_bytes(data)\n    if suffix in {\".txt\", \".md\", \".markdown\", \".csv\"}:\n        return data.decode(\"utf-8\", errors=\"replace\")\n    return data.decode(\"utf-8\", errors=\"replace\")\n\n\ndef read_uploaded_file(uploaded_file: BinaryIO) -> str:\n    name = getattr(uploaded_file, \"name\", \"uploaded.txt\")\n    data = uploaded_file.read()\n    return extract_text_from_bytes(data, name)\n",
  "journal_platform/services.py": "from __future__ import annotations\n\nimport re\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .ai import extract_top_terms, plagiarism_scan, match_reviewers, word_count\nfrom .config import DB_PATH, TOP_REVIEWERS\nfrom .db import execute, fetch_all, fetch_one, initialize_with_seed_data, upsert_user\n\n\ndef slugify(text: str, max_len: int = 70) -> str:\n    text = re.sub(r\"[^a-zA-Z0-9]+\", \"-\", text.lower()).strip(\"-\")\n    return (text[:max_len].strip(\"-\") or \"paper\")\n\n\ndef setup_platform(db_path: Path | str = DB_PATH, reset: bool = False) -> dict[str, int]:\n    return initialize_with_seed_data(db_path, reset=reset)\n\n\ndef get_reviewers(db_path: Path | str = DB_PATH) -> list[dict[str, Any]]:\n    return fetch_all(db_path, \"SELECT * FROM users WHERE role = 'reviewer' ORDER BY name\")\n\n\ndef get_local_similarity_corpus(db_path: Path | str = DB_PATH, exclude_manuscript_id: int | None = None) -> list[dict[str, Any]]:\n    sources = fetch_all(\n        db_path,\n        \"SELECT title, source_type, source_text, url FROM plagiarism_sources ORDER BY created_at DESC\",\n    )\n    params: tuple[Any, ...] = ()\n    where = \"WHERE status IN ('submitted', 'under_review', 'accepted', 'published')\"\n    if exclude_manuscript_id is not None:\n        where += \" AND manuscript_id != ?\"\n        params = (exclude_manuscript_id,)\n    manuscripts = fetch_all(\n        db_path,\n        f\"\"\"\n        SELECT title, status AS source_type, manuscript_text AS source_text, '' AS url\n        FROM manuscripts\n        {where}\n        ORDER BY updated_at DESC\n        \"\"\",\n        params,\n    )\n    return sources + manuscripts\n\n\ndef save_draft(\n    db_path: Path | str,\n    student_name: str,\n    student_email: str,\n    country: str,\n    title: str,\n    abstract: str,\n    field: str,\n    keywords: str,\n    manuscript_text: str,\n    file_name: str = \"online-editor\",\n) -> int:\n    author_id = upsert_user(db_path, student_name, student_email, \"student\", country=country)\n    manuscript_id = execute(\n        db_path,\n        \"\"\"\n        INSERT INTO manuscripts\n        (title, abstract, field, keywords, author_user_id, author_name, country, status, manuscript_text, file_name)\n        VALUES (?, ?, ?, ?, ?, ?, ?, 'draft', ?, ?)\n        \"\"\",\n        (title, abstract, field, keywords, author_id, student_name, country, manuscript_text, file_name),\n    )\n    execute(\n        db_path,\n        \"INSERT INTO manuscript_versions (manuscript_id, version_label, manuscript_text) VALUES (?, ?, ?)\",\n        (manuscript_id, \"draft\", manuscript_text),\n    )\n    return manuscript_id\n\n\ndef submit_manuscript(\n    db_path: Path | str,\n    student_name: str,\n    student_email: str,\n    country: str,\n    title: str,\n    abstract: str,\n    field: str,\n    keywords: str,\n    manuscript_text: str,\n    file_name: str = \"online-editor\",\n    plagiarism_override: dict | None = None,\n) -> dict[str, Any]:\n    author_id = upsert_user(db_path, student_name, student_email, \"student\", country=country)\n    corpus = get_local_similarity_corpus(db_path)\n    scan = plagiarism_override or plagiarism_scan(manuscript_text, corpus, top_n=5)\n    manuscript_id = execute(\n        db_path,\n        \"\"\"\n        INSERT INTO manuscripts\n        (title, abstract, field, keywords, author_user_id, author_name, country, status, manuscript_text,\n         file_name, plagiarism_score, plagiarism_report, submitted_at)\n        VALUES (?, ?, ?, ?, ?, ?, ?, 'submitted', ?, ?, ?, ?, CURRENT_TIMESTAMP)\n        \"\"\",\n        (\n            title,\n            abstract,\n            field,\n            keywords,\n            author_id,\n            student_name,\n            country,\n            manuscript_text,\n            file_name,\n            float(scan[\"score\"]),\n            scan[\"report\"],\n        ),\n    )\n    execute(\n        db_path,\n        \"INSERT INTO manuscript_versions (manuscript_id, version_label, manuscript_text) VALUES (?, ?, ?)\",\n        (manuscript_id, \"submitted\", manuscript_text),\n    )\n    assignments = assign_reviewers(db_path, manuscript_id, top_n=TOP_REVIEWERS)\n    matched_summary = \"; \".join([f\"{a['name']} ({a['match_score']:.1%})\" for a in assignments])\n    execute(\n        db_path,\n        \"UPDATE manuscripts SET status = 'under_review', matched_reviewers = ?, updated_at = CURRENT_TIMESTAMP WHERE manuscript_id = ?\",\n        (matched_summary, manuscript_id),\n    )\n    return {\"manuscript_id\": manuscript_id, \"plagiarism\": scan, \"assignments\": assignments}\n\n\ndef assign_reviewers(db_path: Path | str, manuscript_id: int, top_n: int = TOP_REVIEWERS) -> list[dict[str, Any]]:\n    manuscript = fetch_one(db_path, \"SELECT * FROM manuscripts WHERE manuscript_id = ?\", (manuscript_id,))\n    if manuscript is None:\n        raise ValueError(f\"Unknown manuscript_id: {manuscript_id}\")\n    reviewers = get_reviewers(db_path)\n    matches = match_reviewers(\n        title=manuscript[\"title\"],\n        abstract=manuscript[\"abstract\"] or \"\",\n        field=manuscript[\"field\"] or \"\",\n        keywords=manuscript[\"keywords\"] or \"\",\n        manuscript_text=manuscript[\"manuscript_text\"] or \"\",\n        reviewers=reviewers,\n        top_n=top_n,\n    )\n    for match in matches:\n        link = f\"?assignment_id={manuscript_id}-{match['user_id']}\"\n        execute(\n            db_path,\n            \"\"\"\n            INSERT OR IGNORE INTO review_assignments\n            (manuscript_id, reviewer_user_id, match_score, assignment_link)\n            VALUES (?, ?, ?, ?)\n            \"\"\",\n            (manuscript_id, match[\"user_id\"], float(match[\"match_score\"]), link),\n        )\n        match[\"assignment_link\"] = link\n    return matches\n\n\ndef list_manuscripts(db_path: Path | str = DB_PATH, status: str | None = None) -> list[dict[str, Any]]:\n    if status:\n        return fetch_all(db_path, \"SELECT * FROM manuscripts WHERE status = ? ORDER BY updated_at DESC\", (status,))\n    return fetch_all(db_path, \"SELECT * FROM manuscripts ORDER BY updated_at DESC\")\n\n\ndef list_published(db_path: Path | str = DB_PATH, query: str = \"\") -> list[dict[str, Any]]:\n    query = (query or \"\").strip().lower()\n    rows = fetch_all(db_path, \"SELECT * FROM manuscripts WHERE status = 'published' ORDER BY published_at DESC\")\n    if not query:\n        return rows\n    return [\n        r\n        for r in rows\n        if query in (r.get(\"title\") or \"\").lower()\n        or query in (r.get(\"abstract\") or \"\").lower()\n        or query in (r.get(\"keywords\") or \"\").lower()\n        or query in (r.get(\"author_name\") or \"\").lower()\n    ]\n\n\ndef get_assignments_for_reviewer(db_path: Path | str, reviewer_email: str) -> list[dict[str, Any]]:\n    reviewer_email = reviewer_email.strip().lower()\n    return fetch_all(\n        db_path,\n        \"\"\"\n        SELECT a.*, m.title, m.abstract, m.field, m.keywords, m.manuscript_text, m.author_name,\n               m.country, m.status AS manuscript_status, u.name AS reviewer_name, u.email AS reviewer_email\n        FROM review_assignments a\n        JOIN manuscripts m ON m.manuscript_id = a.manuscript_id\n        JOIN users u ON u.user_id = a.reviewer_user_id\n        WHERE lower(u.email) = ?\n        ORDER BY a.created_at DESC\n        \"\"\",\n        (reviewer_email,),\n    )\n\n\ndef add_review(\n    db_path: Path | str,\n    manuscript_id: int,\n    reviewer_name: str,\n    reviewer_email: str,\n    recommendation: str,\n    comments_to_author: str,\n    comments_to_editor: str = \"\",\n    confidence: int = 3,\n) -> int:\n    reviewer_id = upsert_user(db_path, reviewer_name, reviewer_email, \"reviewer\")\n    review_id = execute(\n        db_path,\n        \"\"\"\n        INSERT INTO reviews\n        (manuscript_id, reviewer_user_id, reviewer_name, reviewer_email, recommendation,\n         comments_to_author, comments_to_editor, confidence)\n        VALUES (?, ?, ?, ?, ?, ?, ?, ?)\n        \"\"\",\n        (\n            manuscript_id,\n            reviewer_id,\n            reviewer_name,\n            reviewer_email.strip().lower(),\n            recommendation,\n            comments_to_author,\n            comments_to_editor,\n            confidence,\n        ),\n    )\n    execute(\n        db_path,\n        \"\"\"\n        UPDATE review_assignments\n        SET status = 'completed'\n        WHERE manuscript_id = ? AND reviewer_user_id = ?\n        \"\"\",\n        (manuscript_id, reviewer_id),\n    )\n    next_status = {\n        \"accept\": \"accepted\",\n        \"minor_revision\": \"minor_revision\",\n        \"major_revision\": \"major_revision\",\n        \"reject\": \"rejected\",\n    }[recommendation]\n    execute(\n        db_path,\n        \"UPDATE manuscripts SET status = ?, updated_at = CURRENT_TIMESTAMP WHERE manuscript_id = ?\",\n        (next_status, manuscript_id),\n    )\n    return review_id\n\n\ndef list_reviews(db_path: Path | str, manuscript_id: int) -> list[dict[str, Any]]:\n    return fetch_all(db_path, \"SELECT * FROM reviews WHERE manuscript_id = ? ORDER BY created_at DESC\", (manuscript_id,))\n\n\ndef publish_manuscript(db_path: Path | str, manuscript_id: int) -> str:\n    manuscript = fetch_one(db_path, \"SELECT * FROM manuscripts WHERE manuscript_id = ?\", (manuscript_id,))\n    if manuscript is None:\n        raise ValueError(f\"Unknown manuscript_id: {manuscript_id}\")\n    base_slug = slugify(manuscript[\"title\"])\n    doi_slug = f\"ahsj-2026-{manuscript_id}-{base_slug}\"\n    execute(\n        db_path,\n        \"\"\"\n        UPDATE manuscripts\n        SET status = 'published', doi_slug = ?, published_at = CURRENT_TIMESTAMP, updated_at = CURRENT_TIMESTAMP\n        WHERE manuscript_id = ?\n        \"\"\",\n        (doi_slug, manuscript_id),\n    )\n    return doi_slug\n\n\ndef update_status(db_path: Path | str, manuscript_id: int, status: str) -> None:\n    execute(db_path, \"UPDATE manuscripts SET status = ?, updated_at = CURRENT_TIMESTAMP WHERE manuscript_id = ?\", (status, manuscript_id))\n\n\ndef manuscript_stats(db_path: Path | str = DB_PATH) -> dict[str, Any]:\n    rows = fetch_all(db_path, \"SELECT status, COUNT(*) AS n FROM manuscripts GROUP BY status\")\n    stats = {row[\"status\"]: row[\"n\"] for row in rows}\n    stats[\"reviewers\"] = fetch_one(db_path, \"SELECT COUNT(*) AS n FROM users WHERE role = 'reviewer'\")[\"n\"]\n    stats[\"students\"] = fetch_one(db_path, \"SELECT COUNT(*) AS n FROM users WHERE role = 'student'\")[\"n\"]\n    return stats\n\n\ndef manuscript_quality_hints(text: str) -> list[str]:\n    hints = []\n    wc = word_count(text)\n    if wc < 500:\n        hints.append(\"The manuscript is short for a research paper. Consider expanding methods, results, and discussion.\")\n    lowered = (text or \"\").lower()\n    for section in [\"abstract\", \"introduction\", \"methods\", \"results\", \"discussion\", \"references\"]:\n        if section not in lowered:\n            hints.append(f\"Consider adding or clearly labeling a {section.title()} section.\")\n    terms = extract_top_terms(text, limit=6)\n    if terms:\n        hints.append(\"AI keyword suggestions: \" + \", \".join(terms))\n    return hints\n",
  "notebooks/README.md": "# Notebook\n\nOpen `high_school_journal_platform_colab.ipynb` in Google Colab and run cells from top to bottom. The notebook recreates the project, installs dependencies, runs the smoke test, and shows launch options.\n",
  "requirements.txt": "streamlit>=1.49\npandas>=2.2\nnumpy>=2.0\nscikit-learn>=1.5\npypdf>=5.0\npython-docx>=1.1\npytest>=8.0\n",
  "schema.sql": "PRAGMA foreign_keys = ON;\n\nCREATE TABLE IF NOT EXISTS users (\n    user_id INTEGER PRIMARY KEY AUTOINCREMENT,\n    name TEXT NOT NULL,\n    email TEXT UNIQUE NOT NULL,\n    role TEXT NOT NULL CHECK (role IN ('student', 'reviewer', 'admin')),\n    country TEXT,\n    expertise TEXT,\n    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP\n);\n\nCREATE TABLE IF NOT EXISTS manuscripts (\n    manuscript_id INTEGER PRIMARY KEY AUTOINCREMENT,\n    title TEXT NOT NULL,\n    abstract TEXT,\n    field TEXT NOT NULL,\n    keywords TEXT,\n    author_user_id INTEGER,\n    author_name TEXT,\n    country TEXT,\n    status TEXT NOT NULL DEFAULT 'draft' CHECK (\n        status IN ('draft', 'submitted', 'under_review', 'minor_revision', 'major_revision', 'accepted', 'rejected', 'published')\n    ),\n    manuscript_text TEXT NOT NULL,\n    file_name TEXT,\n    plagiarism_score REAL NOT NULL DEFAULT 0.0,\n    plagiarism_report TEXT NOT NULL DEFAULT '',\n    matched_reviewers TEXT NOT NULL DEFAULT '',\n    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    submitted_at TEXT,\n    published_at TEXT,\n    doi_slug TEXT UNIQUE,\n    FOREIGN KEY (author_user_id) REFERENCES users(user_id)\n);\n\nCREATE TABLE IF NOT EXISTS manuscript_versions (\n    version_id INTEGER PRIMARY KEY AUTOINCREMENT,\n    manuscript_id INTEGER NOT NULL,\n    version_label TEXT NOT NULL,\n    manuscript_text TEXT NOT NULL,\n    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    FOREIGN KEY (manuscript_id) REFERENCES manuscripts(manuscript_id)\n);\n\nCREATE TABLE IF NOT EXISTS plagiarism_sources (\n    source_id INTEGER PRIMARY KEY AUTOINCREMENT,\n    title TEXT NOT NULL,\n    source_type TEXT NOT NULL DEFAULT 'local',\n    source_text TEXT NOT NULL,\n    url TEXT,\n    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP\n);\n\nCREATE TABLE IF NOT EXISTS review_assignments (\n    assignment_id INTEGER PRIMARY KEY AUTOINCREMENT,\n    manuscript_id INTEGER NOT NULL,\n    reviewer_user_id INTEGER NOT NULL,\n    match_score REAL NOT NULL,\n    status TEXT NOT NULL DEFAULT 'invited' CHECK (status IN ('invited', 'accepted', 'declined', 'completed')),\n    assignment_link TEXT,\n    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    UNIQUE (manuscript_id, reviewer_user_id),\n    FOREIGN KEY (manuscript_id) REFERENCES manuscripts(manuscript_id),\n    FOREIGN KEY (reviewer_user_id) REFERENCES users(user_id)\n);\n\nCREATE TABLE IF NOT EXISTS reviews (\n    review_id INTEGER PRIMARY KEY AUTOINCREMENT,\n    manuscript_id INTEGER NOT NULL,\n    reviewer_user_id INTEGER,\n    reviewer_name TEXT NOT NULL,\n    reviewer_email TEXT,\n    recommendation TEXT NOT NULL CHECK (recommendation IN ('accept', 'minor_revision', 'major_revision', 'reject')),\n    comments_to_author TEXT NOT NULL,\n    comments_to_editor TEXT NOT NULL DEFAULT '',\n    confidence INTEGER NOT NULL DEFAULT 3 CHECK (confidence BETWEEN 1 AND 5),\n    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,\n    FOREIGN KEY (manuscript_id) REFERENCES manuscripts(manuscript_id),\n    FOREIGN KEY (reviewer_user_id) REFERENCES users(user_id)\n);\n",
  "tests/smoke_test.py": "from __future__ import annotations\n\nimport tempfile\nfrom pathlib import Path\nimport sys\n\nROOT = Path(__file__).resolve().parents[1]\nif str(ROOT) not in sys.path:\n    sys.path.insert(0, str(ROOT))\n\nfrom journal_platform.db import fetch_all, initialize_with_seed_data\nfrom journal_platform.services import add_review, list_published, publish_manuscript, submit_manuscript\n\n\ndef run_smoke_test() -> dict:\n    with tempfile.TemporaryDirectory() as tmp:\n        db_path = Path(tmp) / \"smoke.db\"\n        seeded = initialize_with_seed_data(db_path, reset=True)\n        assert seeded[\"reviewers\"] >= 3, \"Expected seeded reviewers\"\n        assert seeded[\"sources\"] >= 3, \"Expected sample corpus\"\n\n        original_text = \"\"\"\n        Abstract: This student research paper studies solar energy for school laboratories.\n        Introduction: Renewable energy can help classrooms understand electricity and sustainability.\n        Methods: We measured voltage from small solar panels under shade and sunlight.\n        Results: The panel produced higher voltage in direct sunlight than under shade.\n        Discussion: The experiment can be improved with repeated trials and temperature records.\n        References: Teacher-provided laboratory manual.\n        \"\"\"\n        result = submit_manuscript(\n            db_path,\n            student_name=\"Test Student\",\n            student_email=\"student@example.org\",\n            country=\"Kenya\",\n            title=\"Solar Panel Voltage in School Laboratories\",\n            abstract=\"A student project measuring small solar panel voltage.\",\n            field=\"Energy\",\n            keywords=\"solar energy, voltage, school laboratory\",\n            manuscript_text=original_text,\n            file_name=\"solar.txt\",\n        )\n        assert result[\"manuscript_id\"] > 0\n        assert len(result[\"assignments\"]) > 0, \"Expected reviewer assignments\"\n\n        copied_result = submit_manuscript(\n            db_path,\n            student_name=\"Second Student\",\n            student_email=\"second@example.org\",\n            country=\"Ghana\",\n            title=\"Copied Solar Panel Voltage Study\",\n            abstract=\"A very similar project.\",\n            field=\"Energy\",\n            keywords=\"solar energy, voltage\",\n            manuscript_text=original_text,\n            file_name=\"copy.txt\",\n        )\n        assert copied_result[\"plagiarism\"][\"score\"] > 0.40, \"Expected high similarity to first submitted manuscript\"\n\n        reviewer_email = result[\"assignments\"][0][\"email\"]\n        review_id = add_review(\n            db_path,\n            manuscript_id=result[\"manuscript_id\"],\n            reviewer_name=result[\"assignments\"][0][\"name\"],\n            reviewer_email=reviewer_email,\n            recommendation=\"accept\",\n            comments_to_author=\"Good student-level study. Please polish the references before publication.\",\n            comments_to_editor=\"Ready for MVP acceptance.\",\n            confidence=4,\n        )\n        assert review_id > 0\n        slug = publish_manuscript(db_path, result[\"manuscript_id\"])\n        assert slug.startswith(\"ahsj-2026\")\n        published = list_published(db_path)\n        assert len(published) == 1\n        assignments = fetch_all(db_path, \"SELECT * FROM review_assignments\")\n        assert assignments, \"Expected assignments in database\"\n        return {\n            \"seeded\": seeded,\n            \"first_manuscript_id\": result[\"manuscript_id\"],\n            \"copy_similarity_score\": round(copied_result[\"plagiarism\"][\"score\"], 3),\n            \"review_id\": review_id,\n            \"published_slug\": slug,\n            \"assignment_count\": len(assignments),\n        }\n\n\nif __name__ == \"__main__\":\n    summary = run_smoke_test()\n    print(\"SMOKE TEST PASSED\")\n    for key, value in summary.items():\n        print(f\"{key}: {value}\")\n"
}

root = Path(PROJECT_NAME)
if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True)

for rel_path, content in PROJECT_FILES.items():
    target = root / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

print(f"Wrote {len(PROJECT_FILES)} files to {root.resolve()}")
print("Next: install dependencies and run the smoke test.")



## Install dependencies

This uses `requirements.txt` from the generated project. Streamlit and scikit-learn may take a minute to install in a new runtime.


In [ ]:

%cd african_high_school_journal_platform
!python --version
!python -m pip install -q -r requirements.txt



## Run the smoke test

The smoke test creates a temporary database, seeds reviewers and sample source texts, submits a manuscript, verifies reviewer matching, submits a duplicate-like manuscript, checks plagiarism similarity, adds a review, and publishes the accepted paper.


In [ ]:

!python tests/smoke_test.py



## Launch Streamlit locally inside the notebook runtime

For local Jupyter, run the first command and open the URL it prints. In Colab, direct browser access to the runtime is not always available, so use the optional tunnel cell below for demos.


In [ ]:

# Local/runtime launch. Stop it from the notebook runtime if needed.
# For a local machine, uncomment the next line:
# !streamlit run app.py



## Optional: public demo URL from Colab using ngrok

Colab is useful for demos, but it is not stable long-term hosting. For a real public MVP, push this repository to GitHub and deploy it on Streamlit Community Cloud.

To use this cell, create an ngrok account and paste your authtoken when prompted. Keep the token private.


In [ ]:

RUN_TUNNEL = False  # Change to True only when you want a temporary public Colab demo URL.

if RUN_TUNNEL:
    import getpass
    import subprocess
    import time
    from pathlib import Path

    !python -m pip install -q pyngrok
    from pyngrok import ngrok

    token = getpass.getpass("Paste ngrok authtoken: ")
    if token:
        ngrok.set_auth_token(token)

    log_path = Path("streamlit.log")
    process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
        stdout=log_path.open("w"),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)
    public_url = ngrok.connect(8501)
    print("Streamlit public demo URL:", public_url)
    print("Streamlit process id:", process.pid)
else:
    print("Tunnel not started. Set RUN_TUNNEL = True to create a temporary public demo URL.")



## Create a GitHub-ready zip from the notebook

Run this cell after making edits. It creates `african_high_school_journal_platform.zip`, which you can download from the Colab file browser and upload to GitHub.


In [ ]:

from pathlib import Path
import zipfile

project_root = Path.cwd()
zip_path = project_root.parent / "african_high_school_journal_platform.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in project_root.rglob("*"):
        if path.is_file() and "__pycache__" not in str(path):
            zf.write(path, arcname=Path(project_root.name) / path.relative_to(project_root))
print("Created", zip_path.resolve())



## Deployment checklist

1. Create a GitHub repository for the project.
2. Upload the generated project folder or the zip contents.
3. On Streamlit Community Cloud, create a new app from the GitHub repo.
4. Set the app entrypoint to `app.py`.
5. Add secrets only if you later integrate email, authentication, cloud storage, or a plagiarism API.
6. Before real student use, add real authentication, privacy/consent language, data deletion, reviewer conflict-of-interest rules, and moderation/safety procedures.
